# GPS Track Extractor from GPSMapCamera Video
Extracts continuous lat/lon data from GPS Map Camera overlay burned into MP4 video frames using OCR.

**Video:** `Eng_clg_60504PMByGPSMapCamera.mp4`  
**Location:** Lucknow, Uttar Pradesh, India  
**Date:** 20/05/2026 ~06:05 PM IST

## Step 1: Install Dependencies

In [ ]:
# Install Python packages
import subprocess, sys

packages = ['opencv-python-headless', 'pytesseract', 'Pillow', 'pandas', 'numpy', 'folium', 'tqdm']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ Python packages installed')

# NOTE: Tesseract OCR engine must be installed separately on your OS:
# Ubuntu/Debian:   sudo apt-get install tesseract-ocr
# macOS:           brew install tesseract
# Windows:         https://github.com/UB-Mannheim/tesseract/wiki
print('⚠️  Make sure Tesseract OCR is installed on your system!')

## Step 2: Import Libraries

In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image, ImageEnhance
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully')

## Step 3: Configuration

In [ ]:
# ─── CONFIGURE THESE ───────────────────────────────────────────────────
VIDEO_PATH   = 'Eng_clg_60504PMByGPSMapCamera.mp4'  # path to your video
OUTPUT_CSV   = 'gps_track.csv'                        # output CSV filename
FRAMES_DIR   = 'extracted_frames'                     # temp folder for frames
FPS_EXTRACT  = 1                                       # frames per second to extract (1 = 1 per second)
# ────────────────────────────────────────────────────────────────────────

os.makedirs(FRAMES_DIR, exist_ok=True)
print(f'✅ Config ready. Frames will be saved to: {FRAMES_DIR}/')

## Step 4: Inspect Video

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
duration     = total_frames / fps
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

print(f'📹 Video info:')
print(f'   Resolution : {width} x {height}')
print(f'   FPS        : {fps:.2f}')
print(f'   Duration   : {duration:.1f} seconds ({duration/60:.1f} min)')
print(f'   Total frames: {total_frames}')
print(f'\n→ Will extract ~{int(duration * FPS_EXTRACT)} frames at {FPS_EXTRACT} fps')

## Step 5: Extract Frames (Bottom Third — GPS Overlay Region)

In [ ]:
import subprocess

# Use ffmpeg to extract 1 frame per second, cropping only the bottom third
# where GPSMapCamera renders its overlay
cmd = [
    'ffmpeg', '-i', VIDEO_PATH,
    '-vf', f'fps={FPS_EXTRACT},crop=iw:ih/3:0:2*ih/3',
    os.path.join(FRAMES_DIR, 'frame_%04d.jpg'),
    '-y', '-loglevel', 'error'
]

print('⏳ Extracting frames (cropped to GPS overlay region)...')
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('❌ ffmpeg error:', result.stderr)
else:
    frame_count = len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
    print(f'✅ Extracted {frame_count} frames to {FRAMES_DIR}/')

## Step 6: Preview a Frame

In [ ]:
import matplotlib.pyplot as plt

# Show the first frame to verify GPS overlay is visible
sample = sorted(os.listdir(FRAMES_DIR))[0]
img = Image.open(os.path.join(FRAMES_DIR, sample))

plt.figure(figsize=(10, 3))
plt.imshow(img)
plt.title('Sample extracted frame (GPS overlay region)')
plt.axis('off')
plt.tight_layout()
plt.show()
print('→ Verify you can see: Lat XX.XXXXXX° Long XX.XXXXXX° in the overlay')

## Step 7: OCR — Extract GPS Coordinates from Every Frame

In [ ]:
# Regex patterns for GPS overlay format used by GPSMapCamera
LAT_LON_PATTERN = re.compile(
    r'[Ll]at\s*[:\-]?\s*([+\-]?\d{1,3}[.,]\d+)[°]?\s*[Ll]o?n?g?\s*[:\-]?\s*([+\-]?\d{1,3}[.,]\d+)[°]?'
)
DT_PATTERN = re.compile(
    r'(\w+,\s*\d{1,2}/\d{2}/\d{4}\s+\d{1,2}:\d{2}:\d{2}\s*[AP]M(?:\s*GMT\s*[+\-]\d{2}:\d{2})?)',
    re.IGNORECASE
)

frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
records = []

for i, fname in enumerate(tqdm(frame_files, desc='OCR frames')):
    path = os.path.join(FRAMES_DIR, fname)
    second = i + 1
    
    # Load and enhance image for better OCR accuracy
    img = Image.open(path)
    img = ImageEnhance.Contrast(img).enhance(2.0)  # boost contrast
    img = img.convert('L')                          # convert to grayscale
    
    # Run Tesseract OCR
    text = pytesseract.image_to_string(img, config='--psm 6 --oem 3')
    
    lat, lon, dt_str = None, None, ''
    
    # Extract lat/lon
    m = LAT_LON_PATTERN.search(text)
    if m:
        lat = float(m.group(1).replace(',', '.'))
        lon = float(m.group(2).replace(',', '.'))
    
    # Extract datetime
    dm = DT_PATTERN.search(text)
    if dm:
        dt_str = dm.group(1).strip()
    
    records.append({'second': second, 'latitude': lat, 'longitude': lon, 'datetime_IST': dt_str})

df_raw = pd.DataFrame(records)
success = df_raw['latitude'].notna().sum()
print(f'\n✅ OCR complete: {success}/{len(df_raw)} frames had GPS data ({success/len(df_raw)*100:.1f}%)')

## Step 8: Clean & Interpolate Missing Values

In [ ]:
df = df_raw.copy()

# Convert to numeric (coerce errors to NaN)
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Remove obvious outliers (coordinates outside India's Lucknow region)
df.loc[df['latitude']  < 26.0, 'latitude']  = np.nan
df.loc[df['latitude']  > 28.0, 'latitude']  = np.nan
df.loc[df['longitude'] < 80.0, 'longitude'] = np.nan
df.loc[df['longitude'] > 82.0, 'longitude'] = np.nan

# Linear interpolation for any missing readings
df['latitude']  = df['latitude'].interpolate(method='linear')
df['longitude'] = df['longitude'].interpolate(method='linear')

# Round to 6 decimal places (~0.1m precision)
df['latitude']  = df['latitude'].round(6)
df['longitude'] = df['longitude'].round(6)

print('GPS Track Summary:')
print(f'  Start point : {df["latitude"].iloc[0]}°N, {df["longitude"].iloc[0]}°E')
print(f'  End point   : {df["latitude"].iloc[-1]}°N, {df["longitude"].iloc[-1]}°E')
print(f'  Lat range   : {df["latitude"].min():.6f} → {df["latitude"].max():.6f}')
print(f'  Lon range   : {df["longitude"].min():.6f} → {df["longitude"].max():.6f}')
print(f'  Track points: {len(df)}')
df.head(10)

## Step 9: Save to CSV

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'✅ GPS track saved to: {OUTPUT_CSV}')
print(f'   Rows: {len(df)}')
print(f'   Columns: {list(df.columns)}')

# Preview the CSV
print('\nFirst 5 rows:')
print(df.head().to_string())
print('\nLast 5 rows:')
print(df.tail().to_string())

## Step 10: Visualize GPS Track on Interactive Map

In [ ]:
import folium

# Centre map on mean position
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='OpenStreetMap')

# Draw the route as a polyline
coords = list(zip(df['latitude'], df['longitude']))
folium.PolyLine(coords, color='blue', weight=4, opacity=0.8, tooltip='GPS Track').add_to(m)

# Mark start and end
folium.Marker(
    coords[0],
    popup=f'START\nLat: {coords[0][0]}\nLon: {coords[0][1]}',
    icon=folium.Icon(color='green', icon='play')
).add_to(m)

folium.Marker(
    coords[-1],
    popup=f'END\nLat: {coords[-1][0]}\nLon: {coords[-1][1]}',
    icon=folium.Icon(color='red', icon='stop')
).add_to(m)

# Add circle markers for each point
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color='blue',
        fill=True,
        fill_opacity=0.6,
        tooltip=f"t={row['second']}s | {row['latitude']}, {row['longitude']}"
    ).add_to(m)

m.save('gps_track_map.html')
print('✅ Interactive map saved to: gps_track_map.html')
m  # Display inline in Jupyter

## Step 11: Plot Latitude & Longitude Over Time

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df['second'], df['latitude'],  color='steelblue',  linewidth=1.5, marker='o', markersize=3)
axes[0].set_ylabel('Latitude (°N)')
axes[0].set_title('GPS Track Over Time — Lucknow, UP, India')
axes[0].grid(True, alpha=0.4)

axes[1].plot(df['second'], df['longitude'], color='darkorange', linewidth=1.5, marker='o', markersize=3)
axes[1].set_ylabel('Longitude (°E)')
axes[1].set_xlabel('Time (seconds)')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('gps_timeseries.png', dpi=150)
plt.show()
print('✅ Time-series plot saved to: gps_timeseries.png')

## Step 12: Calculate Distance Travelled

In [ ]:
from math import radians, cos, sin, asin, sqrt

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres between two GPS points."""
    R = 6371000  # Earth radius in metres
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

# Calculate segment distances
distances = [0.0]
for i in range(1, len(df)):
    d = haversine(df['latitude'].iloc[i-1], df['longitude'].iloc[i-1],
                  df['latitude'].iloc[i],   df['longitude'].iloc[i])
    distances.append(d)

df['segment_dist_m']    = distances
df['cumulative_dist_m'] = df['segment_dist_m'].cumsum()

total_distance_m  = df['cumulative_dist_m'].iloc[-1]
total_distance_km = total_distance_m / 1000
avg_speed_kmh     = total_distance_km / (len(df) / 3600)

print(f'📍 Total distance   : {total_distance_m:.1f} m  ({total_distance_km:.3f} km)')
print(f'⏱️  Duration         : {len(df)} seconds')
print(f'🚗 Avg speed        : {avg_speed_kmh:.1f} km/h')

# Save enriched CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f'\n✅ Enriched CSV (with distances) saved to: {OUTPUT_CSV}')

## Step 13: Final Output — All 69 GPS Points

In [ ]:
pd.set_option('display.max_rows', 70)
df[['second','latitude','longitude','datetime_IST','cumulative_dist_m']]